# 🌦️ Notebook 02: Exploratory Data Analysis (Basic)

## Overview
This notebook covers fundamental EDA:
1. Global temperature trends over time
2. Monthly temperature distribution (seasonality)
3. Top cities by precipitation
4. Feature correlation matrix
5. Precipitation patterns by season

**Input:** `data/weather_cleaned.csv`  
**Output:** Visualizations saved to `reports/figures/`

---

## Step 1: Import Libraries & Load Data

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
import warnings
warnings.filterwarnings('ignore')

# Set style
sns.set_style("darkgrid")
plt.rcParams['figure.figsize'] = (14, 6)

print("✅ Libraries imported successfully")

In [ ]:
# Load cleaned data
df = pd.read_csv("../data/weather_cleaned.csv", parse_dates=["last_updated"])

print(f"✅ Loaded cleaned dataset")
print(f"   Shape: {df.shape}")
print(f"   Date range: {df['last_updated'].min()} to {df['last_updated'].max()}")

## Analysis 1: Global Temperature Trends

In [ ]:
# Calculate daily global average temperature
daily_avg = df.groupby("last_updated")["temperature_celsius"].mean().reset_index()

# Create interactive line plot
fig = px.line(
    daily_avg, 
    x="last_updated", 
    y="temperature_celsius",
    title="Global Average Daily Temperature Over Time",
    labels={"temperature_celsius": "Avg Temp (°C)", "last_updated": "Date"},
    template="plotly_dark",
    height=500
)
fig.update_traces(line_color="#FF6B35", line_width=2)
fig.write_html("../reports/figures/01_global_temp_trend.html")
fig.show()

print(f"✅ Temperature trend analysis:")
print(f"   Mean: {daily_avg['temperature_celsius'].mean():.2f}°C")
print(f"   Std Dev: {daily_avg['temperature_celsius'].std():.2f}°C")
print(f"   Min: {daily_avg['temperature_celsius'].min():.2f}°C")
print(f"   Max: {daily_avg['temperature_celsius'].max():.2f}°C")

## Analysis 2: Monthly Temperature Distribution

In [ ]:
# Create boxplot by month (shows seasonality)
fig, ax = plt.subplots(figsize=(14, 6))
sns.boxplot(
    data=df, 
    x="month", 
    y="temperature_celsius",
    palette="coolwarm", 
    ax=ax
)
ax.set_title("Monthly Temperature Distribution (Global)", fontsize=14, fontweight="bold")
ax.set_xlabel("Month", fontsize=12)
ax.set_ylabel("Temperature (°C)", fontsize=12)
plt.tight_layout()
plt.savefig("../reports/figures/02_monthly_temp_boxplot.png", dpi=150, bbox_inches="tight")
plt.show()

print(f"✅ Monthly temperature analysis:")
monthly_stats = df.groupby("month")["temperature_celsius"].agg(["mean", "std", "min", "max"])
print(monthly_stats.round(2))

## Analysis 3: Top Cities by Precipitation

In [ ]:
# Find top 20 wettest cities
top_rain = (
    df.groupby("location_name")["precip_mm"]
    .mean()
    .sort_values(ascending=False)
    .head(20)
    .reset_index()
)

# Create horizontal bar chart
fig = px.bar(
    top_rain, 
    x="precip_mm", 
    y="location_name", 
    orientation="h",
    title="Top 20 Cities by Average Daily Precipitation",
    color="precip_mm", 
    color_continuous_scale="Blues",
    template="plotly_white",
    height=600
)
fig.update_layout(yaxis={"categoryorder": "total ascending"})
fig.write_html("../reports/figures/03_top_rain_cities.html")
fig.show()

print(f"✅ Top 20 rainiest cities:")
print(top_rain.to_string(index=False))

## Analysis 4: Feature Correlation Matrix

In [ ]:
# Select numeric features for correlation
numeric_features = [
    "temperature_celsius", "humidity", "wind_kph", "precip_mm",
    "pressure_mb", "visibility_km", "uv_index", "cloud",
    "air_quality_us-epa-index", "heat_index", "wind_chill"
]
numeric_features = [c for c in numeric_features if c in df.columns]

# Calculate correlation
corr = df[numeric_features].corr()

# Create heatmap
mask = np.triu(np.ones_like(corr, dtype=bool))
plt.figure(figsize=(12, 10))
sns.heatmap(
    corr, 
    mask=mask, 
    annot=True, 
    fmt=".2f",
    cmap="RdYlGn", 
    center=0, 
    linewidths=0.5,
    annot_kws={"size": 8},
    cbar_kws={"label": "Correlation"}
)
plt.title("Feature Correlation Matrix", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.savefig("../reports/figures/04_correlation_matrix.png", dpi=150, bbox_inches="tight")
plt.show()

print(f"✅ Strong correlations (|r| > 0.5):")
# Find strong correlations
strong_corr = []
for i in range(len(corr.columns)):
    for j in range(i+1, len(corr.columns)):
        if abs(corr.iloc[i, j]) > 0.5:
            strong_corr.append({
                'Feature 1': corr.columns[i],
                'Feature 2': corr.columns[j],
                'Correlation': corr.iloc[i, j]
            })

strong_df = pd.DataFrame(strong_corr).sort_values('Correlation', key=abs, ascending=False)
print(strong_df.to_string(index=False))

## Analysis 5: Precipitation Patterns by Season

In [ ]:
# Create pivot table: season x month
pivot = df.pivot_table(
    values="precip_mm", 
    index="season", 
    columns="month", 
    aggfunc="mean"
)

# Create heatmap
plt.figure(figsize=(12, 4))
sns.heatmap(
    pivot, 
    annot=True, 
    fmt=".1f", 
    cmap="YlGnBu", 
    linewidths=0.5,
    cbar_kws={"label": "Avg Precipitation (mm)"}
)
plt.title("Average Precipitation (mm) by Season and Month", fontsize=12, fontweight="bold")
plt.xlabel("Month")
plt.ylabel("Season")
plt.tight_layout()
plt.savefig("../reports/figures/05_precip_heatmap.png", dpi=150, bbox_inches="tight")
plt.show()

print(f"✅ Precipitation by season:")
print(df.groupby("season")["precip_mm"].agg(["mean", "median", "max"]).round(2))

## Summary Statistics

In [ ]:
print("\n" + "="*60)
print("BASIC EDA SUMMARY")
print("="*60)

print(f"\n📊 Key Findings:")
print(f"   • Global avg temperature: {df['temperature_celsius'].mean():.1f}°C")
print(f"   • Temperature range: {df['temperature_celsius'].min():.1f}°C to {df['temperature_celsius'].max():.1f}°C")
print(f"   • Average precipitation: {df['precip_mm'].mean():.1f}mm")
print(f"   • Most humid month: {df.groupby('month')['humidity'].mean().idxmax()} ({df.groupby('month')['humidity'].mean().max():.1f}%)")
print(f"   • Wettest season: {df.groupby('season')['precip_mm'].mean().idxmax()} ({df.groupby('season')['precip_mm'].mean().max():.1f}mm)")
print(f"\n✅ All visualizations saved to reports/figures/")
print(f"\n🎉 Notebook 02 Complete! Ready for Advanced EDA.")